In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import re
from urllib.parse import urljoin
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
import datetime
from selenium import webdriver
import pdfplumber
from urllib.parse import urlsplit
from pathlib import Path
from time import sleep
from docx import Document
import win32com.client as win32
import zipfile
import xml.etree.ElementTree as ET
import base64
import time
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MD CNPF' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running MD CNPF Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
url = "https://www.cnpf.md/ro/registrele-actelor-permisive-6412.html"
ext_pattern = re.compile(r"\.(pdf|doc|docx|xls|xlsx)$", re.IGNORECASE)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
timeout = 30

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

Typology={

       regulatorName + ' 1': 'Companies licensed or authorized on the capital market',
       regulatorName + ' 2': 'Authorized companies in the stock market',
       regulatorName + ' 3': 'Register of crowdfunding service providers',
       regulatorName + ' 4': 'Entities holding information on securities holders',
       regulatorName + ' 5': 'Issuers of securities that have concluded register keeping contracts with registrars',
       regulatorName + ' 6': 'Issuers of securities whose shares are kept by the Central Single Depository',
       regulatorName + ' 7': 'Investment funds that have reorganized into joint-stock companies',
       regulatorName + ' 8': 'Trust companies',
        }

In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
chromeOptions.add_argument("--ignore-certificate-errors")
chromeOptions.add_argument("--allow-insecure-localhost")
chromeOptions.add_argument("--disable-web-security")  # optional; often not needed
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def normalize_block(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r", "\n").replace("\t", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_first(pattern: str, text: str, group: int = 1) -> str:
    m = re.search(pattern, text, flags=re.IGNORECASE)
    return m.group(group).strip() if m else ""

def extract_between(start_pat: str, end_pat: str, text: str) -> str:
    m = re.search(start_pat + r"(.*?)" + end_pat, text, flags=re.IGNORECASE)
    return m.group(1).strip() if m else ""

def parse_contact_block(raw: str):
    text = normalize_block(raw)

    idno = extract_first(r"\bIDNO\s*:\s*([0-9]{6,})\b", text)

    adresa = extract_between(
        r"\bAdresa\s+juridic[ăa]\s*:\s*",
        r"(?=\b(?:Tel(?:/fax)?|Fax|e-?mail|Pagina\s+web|pagina\s+web|IDNO|Adresa\s+po[șs]tal[ăa])\b|$)",
        text,
    )

    tel = extract_first(
        r"\bTel(?:/fax)?\s*:\s*(.*?)(?=\b(?:Fax|e-?mail|Pagina\s+web|pagina\s+web|IDNO)\b|$)",
        text,
    )

    email = extract_first(
        r"\be-?mail\s*:\s*([A-Z0-9._%+\-]+@[A-Z0-9.\-]+\.[A-Z]{2,})",
        text,
    )

    web = extract_first(
        r"\b(?:Pagina\s+web|pagina\s+web)(?:\s+oficial[ăa])?\s*:\s*([^\s;]+)",
        text,
    )

    return idno, adresa, tel, email, web

def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

resp = requests.get(url, headers=headers, timeout=timeout,verify=False)
resp.raise_for_status()
resp.encoding = resp.apparent_encoding  # helps if site encoding is not utf-8
html = resp.text
print('Status Code, If 200 salary will upupupup: ',resp.status_code)
soup = BeautifulSoup(html, "lxml")
buttons = soup.select("button.accordion")
rows = []

for i, btn in enumerate(buttons, start=1):
    title = " ".join(btn.get_text(" ", strip=True).split())
    sibling = btn.find_next_sibling()  # typical pattern: <button class="accordion"> + <div>...</div>

    if sibling is None:
        continue

    for a in sibling.select("a[href]"):
        href = a.get("href", "").strip()
        if not href:
            continue

        full_url = urljoin(url, href)
        if not ext_pattern.search(full_url.split("?")[0].split("#")[0]):
            continue

        link_text = " ".join(a.get_text(" ", strip=True).split())
        rows.append({
            "accordion_title": title,
            "link_text": link_text,
            "file_url": full_url,
        })

df = pd.DataFrame(rows).drop_duplicates()
preferred = {
    "Entități care deţin informaţia privind deţinătorii de valori mobiliare":
        "Lista persoanelor autorizate",
    "Fonduri de investiţii în proces de lichidare si Fonduri de investiții care s-au reorganizat în Societăți pe acțiuni":
        "Fonduri de investitii care s-au reorganizat",
}

out = []
for acc_title, grp in df.groupby("accordion_title", sort=False):
    if acc_title in preferred:
        key = preferred[acc_title].lower()
        pick = grp[grp["link_text"].str.lower().str.contains(key, na=False)]
        out.append(pick.head(1) if len(pick) else grp.head(1))  # fallback if no match
    else:
        out.append(grp)

df2 = pd.concat(out, ignore_index=True).drop_duplicates()
df = df2.copy()
df["clean_url"] = df["file_url"].map(lambda u: urlsplit(u).path)  # only the path part
df["ext"] = df["clean_url"].map(lambda p: Path(p).suffix.lower())


for idx, (file_url, ext) in enumerate(zip(df["file_url"], df["ext"])):
    if os.path.exists(tempfolder):
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem))
    else:
        os.mkdir(tempfolder)
    print(' ======   INdex  ====== ', idx, ext)
    # if idx != 6:
    driver.get(file_url)
    sleep(5)
    for times in range(50):
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
            break
    if ext == ".pdf":
        print("PDF  ->", file_url)
        if idx ==0:
            table_settings = {
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "intersection_tolerance": 5,
                "snap_tolerance": 3,
                "join_tolerance": 3,
                "edge_min_length": 3,
                "min_words_vertical": 1,
                "min_words_horizontal": 1,
            }
            tables = []  # list of dicts: page_number, table_index, rows
            sleep(5)

            with pdfplumber.open(dl_files[0]) as pdf:
                for page_i, page in enumerate(pdf.pages, start=1):
                    page_tables = page.extract_tables(table_settings=table_settings)
                    for t_i, rows in enumerate(page_tables, start=1):
                        tables.append({"page": page_i, "table": t_i, "rows": rows})
                
                # len(tables)
                for t in tables:
                    for cell in t['rows']:
                        #print(cell)
                        if cell[0] and cell[0].split('.')[0].isdigit():
                            name = cell[1].replace('\n',' ')
                            cnpf_ = cell[-1].replace('\n',' ')
                            idno, adresa, tel, email, web = parse_contact_block(cell[3])
                            # now you have variables; example:
                            #print(name, adresa,idno, email, web,tel)
                            sqldict['Name'].append(name)
                            sqldict['Address_1'].append(adresa)
                            sqldict['InternalID_1'].append(idno)
                            sqldict['InternalID_1_type'].append('IDNO')
                            sqldict['InternalID_2'].append(cnpf_.replace('\n',' '))
                            sqldict['InternalID_2_type'].append('Nr. și data deciziei C.N.P.F. cu privire la înscrierea în Registru')
                            sqldict['Email'].append(email)
                            sqldict['Website'].append(web)
                            sqldict['Phone'].append(tel)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])
                            sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])
                            sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])
                            sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])
                            sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)

                try:
                    os.remove(dl_files[0])
                    
                except PermissionError:
                    sleep(2)
        else:
            tables = []  # list of dicts: page_number, table_index, rows
            sleep(5)
            with pdfplumber.open(dl_files[0]) as pdf:
                for page_i, page in enumerate(pdf.pages, start=1):
                    page_tables = page.extract_tables(table_settings=table_settings)
                    for t_i, rows in enumerate(page_tables, start=1):
                        tables.append({"page": page_i, "table": t_i, "rows": rows})
            for t in tables:
                for cell in t['rows']:
                    #print(cell)
                    if cell[0] and str(cell[0]).split('.')[0].isdigit():
                        name = (cell[1] or '').replace('\n',' ')
                        adresa = (cell[3] or '').replace('\n',' ')
                        email = (re.search(r'[\w\.-]+@[\w\.-]+\.\w+', adresa) or [''])[0]; 
                        addr_no_email = re.sub(r'\s*(?:e-?\s*mail\s*:\s*)?[\w\.-]+@[\w\.-]+\.\w+\s*', ' ', adresa, flags=re.I); 
                        addr_no_email = " ".join(addr_no_email.split())
                        tele_ = (cell[-1] or '').replace('\n',' ')
                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(addr_no_email)
                        sqldict['Email'].append(email)
                        sqldict['Phone'].append(tele_)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])
                        sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])
                        sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])
                        sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])
                        sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
            
                

        # handle pdf
    elif ext in (".xlsx", ".xls"):
        print("XLSX ->", file_url)
        if idx == 2:
            with pd.ExcelFile(dl_files[0]) as xlsx:
                print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --") 
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                data = data.dropna(subset=[data.columns[0], data.columns[1]])
                data.columns = data.iloc[0]
                data = data[1:].reset_index(drop=True)
                print(data.shape)
            for _,item_ in data.iterrows():

                name_ = item_[data.columns[2]]
                idno_= item_[data.columns[4]]
                cdpf_ = item_[data.columns[5]]
                address_ = item_[data.columns[6]]
                phone_ = item_[data.columns[7]]
                web_ = item_[data.columns[8]]

                #print(name_.split(':')[-1],idno_.split('IDNO:')[-1],cdpf_,address_.split(':')[-1],phone_.split(':')[-1],web_)
                sqldict['Name'].append(name_.split(':')[-1])
                sqldict['InternalID_1'].append(idno_.split('IDNO:')[-1])
                sqldict['InternalID_1_type'].append('Data înregistrării de stat și IDNO')
                sqldict['InternalID_2'].append(cdpf_)
                sqldict['InternalID_2_type'].append('Hotărârea CNPF')
                addr_ = " ".join(str(address_.split(':')[-1]).replace("\xa0", " ").split())
                sqldict['Address_1'].append(addr_)
                sqldict['Email'].append(phone_.split(':')[-1])
                sqldict['Website'].append(web_)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])
                sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])
                sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])
                sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
        elif idx ==4:
            with pd.ExcelFile(dl_files[0]) as xlsx:
                print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --") 
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                data = data.dropna(subset=[data.columns[0], data.columns[1]])
                data.columns = data.iloc[0]
                data = data[1:].reset_index(drop=True)
                print(data.shape)
                for _,item_ in data.iterrows():
                    name_ = item_[data.columns[1]]
                    isin_= item_[data.columns[2]]
                    idno_ = item_[data.columns[3]]
                    address_ = item_[data.columns[4]]
                    #print(name_,isin_,idno_,address_)


                    #print(name_.split(':')[-1],idno_.split('IDNO:')[-1],cdpf_,address_.split(':')[-1],phone_.split(':')[-1],web_)
                    sqldict['Name'].append(name_)
                    sqldict['InternalID_1'].append(idno_)
                    sqldict['InternalID_1_type'].append('IDNO')
                    sqldict['InternalID_2'].append(isin_)
                    sqldict['InternalID_2_type'].append('ISIN')
                    sqldict['Address_1'].append(address_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])
                    sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])
                    sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])
                    sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
                            
        elif idx == 5:    
            with pd.ExcelFile(dl_files[0]) as xlsx:
                print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --") 
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                data = data.dropna(subset=[data.columns[0]])
                print(data.shape)
                for _,item_ in data.iterrows():
                    name_ = item_[data.columns[1]]
                    idno_ = item_[data.columns[2]]
                    address_ = item_[data.columns[3]]
                    sqldict['Name'].append(name_)
                    sqldict['InternalID_1'].append(idno_)
                    sqldict['InternalID_1_type'].append('IDNO')
                    sqldict['Address_1'].append(address_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])
                    sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])
                    sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])
                    sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)

    elif ext in (".docx", ".doc"):
        print("DOC  ->", file_url)
        if idx ==1:
            doc = Document(dl_files[0])
            tables = doc.tables
            print(f"[INFO] -- Number of tables: {len(tables)}  --")
            if len(tables) == 0:
                data = pd.DataFrame()
                print(data.shape)
            else:
                # Build DataFrame from the first table (extract cell.text for each cell)
                data = pd.DataFrame([[cell.text for cell in row.cells] for row in tables[0].rows])
                print(data.shape)
                data.columns = data.iloc[0]
                data = data[1:]
            for  name_,interal_id, company_adresa,tele, cnpf_ in zip(
                                                                        data[data.columns[1]], 
                                                                        data[data.columns[2]],
                                                                        data[data.columns[3]],
                                                                        data[data.columns[5]], 
                                                                        data[data.columns[-1]],
                                                                    ):
                #print(name_,interal_id,company_adresa,tele.split('\n')[0],cnpf_)
                sqldict['Name'].append(name_)
                sqldict['InternalID_1'].append(interal_id)
                sqldict['InternalID_1_type'].append('Codul fiscal (IDNO)')
                sqldict['Address_1'].append(company_adresa.replace('\n',' '))
                sqldict['Phone'].append(tele.split('\n')[0])
                sqldict['InternalID_2'].append(cnpf_.replace('\n',' '))
                sqldict['InternalID_2_type'].append('Nr. și data deciziei C.N.P.F. cu privire la înscrierea în Registru')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])
                sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])
                sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])
                sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
        elif idx == 6:

            word = win32.Dispatch("Word.Application")

            word.Visible = False

            doc = word.Documents.Open(dl_files[0])

            doc.SaveAs(dl_files[0], FileFormat=16)  # 16 = wdFormatDocumentDefault (.docx)

            doc.Close()

            word.Quit()

            doc = Document(dl_files[0])

            tables = doc.tables

            print(f"[INFO] -- Number of tables: {len(tables)}  --")

            if len(tables) == 0:

                data = pd.DataFrame()

                print(data.shape)

            else:

                # Build DataFrame from the first table (extract cell.text for each cell)

                data = pd.DataFrame([[cell.text for cell in row.cells] for row in tables[0].rows])

                print(data.shape)

                data.columns = data.iloc[0]

                data = data[1:]

            for  name_, company_adresa in zip(

                                                                                    data[data.columns[2]], 



                                                                                    data[data.columns[4]],



                                                                                ): 

                # Use name_ consistently and properly trim the phone part instead of accessing .text on a method



                    name_new = name_.split('\n')[0]

                    idno_ = name_.split('IDNO')[-1].split('\n')[0].strip()

                    address_ = company_adresa.split('tel')[0].strip()

                    tel_ = company_adresa.split('tel')[-1].lstrip(': ').lstrip('. ').strip()

                    sqldict['Name'].append(name_new)

                    sqldict['InternalID_1'].append(idno_)

                    sqldict['InternalID_1_type'].append('IDNO')

                    sqldict['Address_1'].append(address_)

                    sqldict['Phone'].append(tel_)

                    sqldict['ListProcessDate'].append(processdate)

                    sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])

                    sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])

                    sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])

                    sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])

                    sqldict['RegulationType'].append('Regulated')

                    sqldict = bourange_same_length_array(sqldict)



        elif idx == 7:

            word = win32.Dispatch("Word.Application")

            word.Visible = False

            doc = word.Documents.Open(dl_files[0])

            doc.SaveAs(dl_files[0], FileFormat=16)  # 16 = wdFormatDocumentDefault (.docx)

            doc.Close()

            word.Quit()

            doc = Document(dl_files[0])

            tables = doc.tables

            print(f"[INFO] -- Number of tables: {len(tables)}  --")

            if len(tables) == 0:

                data = pd.DataFrame()

                print(data.shape)

            else:

                # Build DataFrame from the first table (extract cell.text for each cell)

                data = pd.DataFrame([[cell.text for cell in row.cells] for row in tables[0].rows])

                print(data.shape)

                data.columns = data.iloc[0]

                data = data[1:]

            for  name_, company_adresa,tele, cnpf_ in zip(

                                                                        data[data.columns[1]], 

                                                                        data[data.columns[3]],

                                                                        data[data.columns[4]],

                                                                        data[data.columns[-1]],

                                                                    ):

                #print(name_,interal_id,company_adresa,tele.split('\n')[0],cnpf_)

                sqldict['Name'].append(name_)

                sqldict['InternalID_1'].append(cnpf_)

                sqldict['InternalID_1_type'].append('Reprezentantul CNPF/administrator din oficiu')

                sqldict['Address_1'].append(company_adresa.replace('\n',' '))

                sqldict['Phone'].append(tele.split('\n')[0])

                sqldict['ListProcessDate'].append(processdate)

                sqldict['ListName'].append(Typology[f"{regulatorName} {idx+1}"])

                sqldict['RegCtry'].append(f"{regulatorName} {idx+1}".split()[0])

                sqldict['RegCode'].append(f"{regulatorName} {idx+1}".split()[1])

                sqldict['ListCode'].append(f"{regulatorName} {idx+1}".split()[2])

                sqldict['RegulationType'].append('Regulated')

            sqldict = bourange_same_length_array(sqldict)

c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.cnpf.md'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status Code, If 200 salary will upupupup:  200
 ======   INdex  ======  0 .pdf
PDF  -> https://www.cnpf.md/storage/files/files/LISTA%20Participanti_WEB%202%20_pt_SAIT_Posta%2001_07_25_R(1).pdf
 ======   INdex  ======  1 .docx
DOC  -> https://www.cnpf.md/storage/files/files/Registrul_persoanelor_autorizate_23_12_2024_actualizare%2017_06_25.docx
[INFO] -- Number of tables: 2  --
(5, 9)
 ======   INdex  ======  2 .xlsx
XLSX -> https://www.cnpf.md/storage/files/files/Registrul%20public%20al%20furnizorilor%20de%20servicii%20crowdfunding%202025.xlsx
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Sheet1']  --
(2, 13)
 ======   INdex  ======  3 .pdf
PDF  -> https://www.cnpf.md/storage/files/files/Pt%20SAIT%20_Participanti%20Detin%20INFORMATii_01_07_25_Tank_R.pdf
 ======   INdex  ======  4 .xlsx
XLSX -> https://www.cnpf.md/storage/files/files/Contracte_SR%2011_12_2025.xlsx
[INFO] -- Number of sheets: 2  --
[INFO] -- Number of sheets: ['CONTRACTE ACTIVE', 'CONTRACTE REZILIATE AN

In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
#driver.quit()
sleep(3)

In [8]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 1331 values.
Key 'priority' has 1331 values.
Key 'ListLabel' has 1331 values.
Key 'Typology' has 1331 values.
Key 'EntryType' has 1331 values.
Key 'Name' has 1331 values.
Key 'InternalID_1' has 1331 values.
Key 'InternalID_1_type' has 1331 values.
Key 'InternalID_2' has 1331 values.
Key 'InternalID_2_type' has 1331 values.
Key 'InternalID_3' has 1331 values.
Key 'InternalID_3_type' has 1331 values.
Key 'CoType' has 1331 values.
Key 'License_Type' has 1331 values.
Key 'Address_1' has 1331 values.
Key 'Address_2' has 1331 values.
Key 'City' has 1331 values.
Key 'Zip' has 1331 values.
Key 'Cntry' has 1331 values.
Key 'Phone' has 1331 values.
Key 'Fax' has 1331 values.
Key 'Website' has 1331 values.
Key 'Email' has 1331 values.
Key 'RegulationType' has 1331 values.
Key 'RegulationTypeCode' has 1331 values.
Key 'RegulationDate' has 1331 values.
Key 'CancellationDate' has 1331 values.
Key 'RegCtry' has 1331 values.
Key 'RegCode' has 1331 values.
Key 'ListCode' has 1331 values

In [9]:

doc_url = "https://www.cnpf.md/storage/old_site_files/file/Entitati_Supraveghere/2017/Fonduri_investitii_SA_15_11_17.doc"
viewer_url = "https://view.officeapps.live.com/op/view.aspx?src=" + quote(doc_url, safe="")

opts = Options()
opts.add_argument("--ignore-certificate-errors")
driver = webdriver.Chrome(options=opts)

driver.get(viewer_url)
time.sleep(8)  # or WebDriverWait if you prefer

pdf = driver.execute_cdp_cmd("Page.printToPDF", {"printBackground": True})
pdf_bytes = base64.b64decode(pdf["data"])

out_path = "tempfolder/Fonduri_investitii_SA_15_11_17.pdf"
with open(out_path, "wb") as f:
    f.write(pdf_bytes)

driver.quit()
